In [108]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

from gensim.corpora import Dictionary
from gensim.models import word2vec

from sklearn.manifold import TSNE as tsne

import plotly.io as pio
pio.renderers.default = "vscode"

In [109]:
OHCO = ['book_title', 'chap_num', 'para_num', 'sent_num', 'token_num']
BAG = OHCO[:2]

In [110]:
TOKENS=pd.read_csv("Corpus_Tokens.csv").set_index(OHCO)

In [111]:
docs = TOKENS.groupby(BAG).term_str.apply(list).tolist()

In [112]:
w2v_params = dict(window = 5,
    vector_size = 100,
    min_count = 250,
    workers = 4
)

In [113]:
model = word2vec.Word2Vec(docs, **w2v_params)

In [114]:
WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
WV.index.name = 'term_str'
WV

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
term_str,,,,,,,,,,,,,,,,,,,,,
the,-1.110160,0.861097,0.624535,0.227946,0.040547,-0.961663,0.724586,0.280588,0.017533,-0.295909,...,-0.731511,0.424679,-0.631560,-0.318958,-1.240746,-0.513329,-1.054382,-0.799748,0.040526,-0.296469
and,-0.512989,0.093582,0.223332,0.086340,-0.169860,-0.821383,-0.237382,0.338578,0.256460,-0.289470,...,-0.676284,0.495795,-0.203720,-0.681389,-0.905471,0.307812,1.252472,0.270924,0.421192,0.784293
to,0.455574,0.202118,-0.872452,-1.011958,1.110417,-0.382732,-0.654457,0.976182,-0.603973,1.174572,...,0.280523,0.113852,0.154695,0.351057,-1.389179,-0.243755,-0.656988,0.436509,0.656349,-0.605521
of,-0.106060,0.573100,0.267746,-0.032419,0.095526,-1.257094,-0.506687,-0.085967,1.118399,-0.564042,...,-0.152675,0.744853,0.692454,-0.337713,0.215124,-0.228010,1.225502,-1.144635,-0.188895,-1.307271
a,-0.994532,0.177426,-0.787704,1.402451,0.248276,-0.678819,1.087174,0.713360,-0.147965,-0.327277,...,-0.270053,0.097497,-0.180882,-0.159196,-0.430892,-1.871355,-0.060220,-0.143407,1.453627,0.467214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
relief,0.154459,-0.412065,0.041393,0.136989,0.332138,-0.068388,-0.778846,-0.042142,-1.241002,-0.511078,...,-0.512207,0.041069,0.345707,-0.511911,-0.546794,0.080561,0.714495,-0.968577,0.131367,0.694896
especially,0.048215,0.153912,-0.100881,-0.428282,-0.578171,-0.469592,-0.271237,-0.334536,0.405362,-0.172645,...,0.605179,-0.238246,-0.256446,-0.253460,-0.513775,0.162297,0.064270,0.284185,0.278754,0.066805
credit,0.334326,-0.941641,-0.023502,-0.436247,0.244886,0.873907,-0.067083,-0.956191,-0.063108,0.096715,...,-0.868380,-0.066846,-0.594157,-0.214404,-0.283993,-0.785728,0.700444,-0.794416,0.746129,-0.452518


In [115]:
PP = 50

tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='pca', 
    max_iter=2500, 
    random_state=23
)
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(WV), 
    columns=['x','y'], 
    index=WV.index)
TSNE

,x,y
term_str,,
the,5.419756,5.429569
and,-5.703927,6.695040
to,0.477137,-10.894188
of,1.852898,-2.424006
a,-2.510706,2.127542
...,...,...
relief,6.134735,-8.653453
especially,2.559912,1.625090
credit,11.353740,-10.127103


In [116]:
fig = px.scatter(
    TSNE.reset_index(),
    x='x',
    y='y',
    text='term_str',
    hover_name='term_str',
    height=1000,
    width=1200
)

fig.update_traces(
    mode='markers+text',
    textfont=dict(color='black', size=14, family='Arial'),
    textposition='top center'
)


fig.update_layout(
    title=dict(
        text="t-SNE Projection of Word2Vec Embeddings",
        x=0.5,     
        xanchor='center',
        font=dict(size=20)
    )
)

fig.update_layout(
    xaxis_title="x",
    yaxis_title="y"
)

fig.show()

In [117]:
WV.to_csv("VOCAB_W2V.csv")